# Notebook 01 — Data Exploration
**Project:** AAA Northeast Member Analysis  
**Purpose:** Answer five business questions before touching a model  

---
## Questions this notebook must answer
1. What does one row represent? (grain problem)
2. What is our target population after filtering?
3. Which products have sufficient adoption to model?
4. What columns have missing data — and can we recover them?
5. Are any features obvious leakage candidates?

**Rule:** Write a finding under every chart. A chart without a written conclusion is decoration, not analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

from src.config import get_config
cfg = get_config()
print('Config loaded:', cfg.name, 'v' + cfg.version)

## 1. Load Raw Data and Understand the Grain

In [ ]:
df = pd.read_csv(cfg.paths.raw_data, low_memory=False)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print('=== RAW DATA GRAIN ANALYSIS ===')
print(f'Total rows:         {len(df):,}')
print(f'Unique Member Keys: {df["Member Key"].nunique():,}')
print(f'Unique Households:  {df["Household Key"].nunique():,}')
print(f'Columns:            {df.shape[1]}')
print()
print('Rows per member (distribution):')
print(df.groupby('Member Key').size().describe().to_string())

**Finding 1 — Grain:** Each row represents one (member, roadside service call) event. A single member can appear in many rows. This means we must aggregate costs BEFORE deduplicating members, and aggregate members BEFORE modelling at the household level.

## 2. Member Status Distribution

In [ ]:
status_counts = df.drop_duplicates('Member Key')['Member Status'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2E4A8C', '#E84C3D', '#F39C12']
bars = ax.bar(status_counts.index, status_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/sum(status_counts.values):.1%})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Member Status Distribution (unique members)', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, max(status_counts.values) * 1.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(cfg.paths.report_dir + 'member_status.png', bbox_inches='tight')
plt.show()
print(status_counts.to_string())

**Finding 2 — Population:** Write your finding here after running: how many ACTIVE members, what % are CANCELLED, and why we keep ACTIVE + PENDING only.

## 3. Missing Values — Full Audit

In [ ]:
from src.features.preprocessing import audit_missing
from src.evaluation.plots import plot_missing_values

# Deduplicate to member level before auditing missing — row-level duplication
# inflates apparent missingness of columns that are only populated for service calls
df_member = df.drop_duplicates('Member Key').copy()

report = audit_missing(df_member)
print(f'Columns with missing values: {len(report)} of {df_member.shape[1]}')
print()
print(report.head(30).to_string())

plot_missing_values(df_member, top_n=25,
                    save_path=cfg.paths.report_dir + 'missing_values.png')
plt.show()

**Finding 3 — Missing Data:** Write your finding after running: which columns are >50% missing, which are recoverable via imputation, which must be excluded.

## 4. Product Adoption — Which Products Can We Model?

In [ ]:
# Compute adoption at HOUSEHOLD level (correct grain for business decisions)
yn_map = {'Y': 1, 'N': 0}
df_prod = df_member.copy()
for raw_col, name in cfg.data.col_to_name.items():
    if raw_col in df_prod.columns:
        df_prod[name] = df_prod[raw_col].map(yn_map)

df_hh_prod = (
    df_prod.groupby('Household Key')[cfg.data.product_names]
    .max()
    .dropna(how='all')
)

adoption = df_hh_prod.mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2E4A8C' if v >= 0.05 else '#AABDDB' for v in adoption.values]
bars = ax.barh(adoption.index, adoption.values, color=colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, adoption.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=9)
ax.axvline(x=0.05, color='#E84C3D', linestyle='--', lw=1.8,
           alpha=0.8, label='5% modeling threshold')
ax.set_xlabel('Proportion of Households', fontsize=11)
ax.set_title('Product Adoption — Active Households', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(cfg.paths.report_dir + 'product_adoption.png', bbox_inches='tight')
plt.show()

print('\nHousehold-level adoption:')
for prod, rate in adoption.sort_values(ascending=False).items():
    flag = '✅ model' if rate >= 0.05 else '⬜ below threshold'
    print(f'  {prod:<30} {rate:.1%}  {flag}')

**Finding 4 — Modelable Products:** Write which products exceed the 5% threshold after running. These are your classification targets. Products below threshold have too few positives for a reliable model — not enough signal.

## 5. Multi-Product Members — Relationship Depth

In [ ]:
# How many products does each household hold?
product_count = df_hh_prod.sum(axis=1).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: count distribution
axes[0].bar(product_count.index, product_count.values,
            color='#2E4A8C', edgecolor='white')
axes[0].set_xlabel('Number of Products Held', fontsize=11)
axes[0].set_ylabel('Number of Households', fontsize=11)
axes[0].set_title('Household Product Depth', fontsize=12, fontweight='bold')

# Right: cumulative %
cumulative = product_count.sort_index().cumsum() / product_count.sum()
axes[1].plot(cumulative.index, cumulative.values, 'o-',
             color='#2E4A8C', lw=2, markersize=7)
axes[1].axhline(y=0.80, color='#E84C3D', linestyle='--', alpha=0.7, label='80% of households')
axes[1].set_xlabel('Number of Products Held', fontsize=11)
axes[1].set_ylabel('Cumulative % of Households', fontsize=11)
axes[1].set_title('Cumulative Product Distribution', fontsize=12, fontweight='bold')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].legend(fontsize=9)

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Member Relationship Depth', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(cfg.paths.report_dir + 'product_depth.png', bbox_inches='tight')
plt.show()

zero_prod_pct = product_count.get(0, 0) / product_count.sum()
multi_pct     = product_count[product_count.index >= 2].sum() / product_count.sum()
print(f'Households with 0 products: {zero_prod_pct:.1%}')
print(f'Households with 2+ products: {multi_pct:.1%}')

**Finding 5 — Product Depth:** Write your finding after running. What % of households hold zero AAA Financial products? These are your primary cross-sell opportunity.

## 6. ERS Engagement vs. Product Adoption

In [ ]:
# Hypothesis: members who actually used roadside assistance are more engaged
# and therefore more likely to buy additional products

df_ers = df_member.copy()

# Proxy for ERS usage: has any SC Date (service call record)
df_ers['has_used_ers'] = (~df_ers['SC Date'].isna()).astype(int)

# Attach product flags
for raw_col, name in cfg.data.col_to_name.items():
    if raw_col in df_ers.columns:
        df_ers[name] = df_ers[raw_col].map({'Y': 1, 'N': 0})

target_prods = [p for p in cfg.data.product_names if p in df_ers.columns]

# Adoption rate by ERS usage — group comparison
comparison = df_ers.groupby('has_used_ers')[target_prods].mean()
comparison.index = ['No ERS Usage', 'Has Used ERS']

fig, ax = plt.subplots(figsize=(11, 5))
x   = range(len(target_prods))
w   = 0.35
b1  = ax.bar([xi - w/2 for xi in x], comparison.iloc[0], w,
             label='No ERS Usage', color='#AABDDB', edgecolor='white')
b2  = ax.bar([xi + w/2 for xi in x], comparison.iloc[1], w,
             label='Has Used ERS', color='#2E4A8C', edgecolor='white')

ax.set_xticks(list(x))
ax.set_xticklabels(target_prods, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Adoption Rate', fontsize=11)
ax.set_title('Product Adoption Rate by ERS Engagement', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(cfg.paths.report_dir + 'ers_vs_adoption.png', bbox_inches='tight')
plt.show()

print('Adoption rate by ERS usage:')
print(comparison.T.to_string())

**Finding 6 — ERS Signal:** Write your finding after running. Do members who used ERS show higher product adoption rates? This validates has_used_ers as a key engineered feature.

## 7. Income Distribution by Product Adoption

In [ ]:
from src.features.preprocessing import encode_ordinals

df_inc = encode_ordinals(df_member, cfg.data)

# Focus on INS Client (highest adoption) and FSV Credit Card
for raw_col, name in cfg.data.col_to_name.items():
    if raw_col in df_inc.columns:
        df_inc[name] = df_inc[raw_col].map({'Y': 1, 'N': 0})

target_prods = ['INS Client', 'TRV Globalware', 'FSV Credit Card']
target_prods = [p for p in target_prods if p in df_inc.columns]

if target_prods and 'Income' in df_inc.columns:
    fig, axes = plt.subplots(1, len(target_prods), figsize=(5 * len(target_prods), 4), sharey=True)
    if len(target_prods) == 1:
        axes = [axes]

    for ax, prod in zip(axes, target_prods):
        groups = [
            df_inc.loc[df_inc[prod] == 0, 'Income'].dropna(),
            df_inc.loc[df_inc[prod] == 1, 'Income'].dropna(),
        ]
        bp = ax.boxplot(groups, labels=['Non-buyer', 'Buyer'],
                        patch_artist=True,
                        boxprops={'facecolor': '#D5E3F7'},
                        medianprops={'color': '#E84C3D', 'linewidth': 2})
        ax.set_title(prod, fontsize=11, fontweight='bold')
        ax.set_ylabel('Income ($)' if ax == axes[0] else '')
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda y, _: f'${y/1000:.0f}K')
        )
        ax.spines[['top','right']].set_visible(False)

    plt.suptitle('Income Distribution: Buyers vs Non-buyers',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(cfg.paths.report_dir + 'income_by_product.png', bbox_inches='tight')
    plt.show()

**Finding 7 — Income Signal:** Write your finding after running. Is the median income of buyers significantly higher than non-buyers for FSV Credit Card? This validates is_high_income as a key feature.

## 8. ERS Cost Distribution

In [ ]:
# Understand the regression target BEFORE building the regressor
cost_data = df[df['Total Cost'] > 0]['Total Cost']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw distribution (skewed)
axes[0].hist(cost_data, bins=60, color='#2E4A8C', edgecolor='white', alpha=0.85)
axes[0].set_title('Raw Total Cost (non-zero)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Cost ($)')
axes[0].set_ylabel('Count')

# Log-transformed distribution
axes[1].hist(np.log1p(cost_data), bins=60, color='#27AE60', edgecolor='white', alpha=0.85)
axes[1].set_title('Log(1 + Cost) Transform', fontsize=11, fontweight='bold')
axes[1].set_xlabel('log(1 + Cost)')

# Zero-inflation: proportion of members with zero cost
member_costs = df.groupby('Member Key')['Total Cost'].sum()
zero_pct = (member_costs == 0).mean()
axes[2].bar(['Zero Cost', 'Non-Zero Cost'],
            [zero_pct, 1 - zero_pct],
            color=['#E84C3D', '#2E4A8C'], edgecolor='white')
axes[2].set_title('Zero-Inflation (member level)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Proportion of Members')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('ERS Cost Distribution Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(cfg.paths.report_dir + 'cost_distribution.png', bbox_inches='tight')
plt.show()

print(f'Members with zero lifetime ERS cost: {zero_pct:.1%}')
print(f'Non-zero cost stats:')
print(cost_data.describe().to_string())

**Finding 8 — Cost Distribution:** Write your finding after running. What % of members have zero ERS cost? Does the log transform make the distribution more Gaussian? This justifies your regression design decision.

## 9. Correlation Analysis — Leakage Check

In [ ]:
# Check for any remaining features with suspiciously high correlation
# to product targets — signals leakage
from src.features.preprocessing import run_preprocessing

print('Running full preprocessing pipeline...')
X, df_hh = run_preprocessing()
print(f'Feature matrix shape: {X.shape}')
print(f'Households:           {len(df_hh)}')
print(f'Null values in X:     {X.isnull().sum().sum()}')

In [ ]:
from src.evaluation.plots import plot_correlation_matrix

plot_correlation_matrix(df_hh, top_n=20,
                        save_path=cfg.paths.report_dir + 'correlation_matrix.png')
plt.show()

In [ ]:
# Check correlation of engineered features with INS Client (highest adoption product)
target = 'INS Client'
if target in df_hh.columns:
    numeric_cols = df_hh.select_dtypes(include=[np.number]).columns
    corr_with_target = (
        df_hh[numeric_cols]
        .corr()[target]
        .drop(target)
        .abs()
        .sort_values(ascending=False)
        .head(20)
    )

    print(f'Top 20 features correlated with {target}:')
    print(corr_with_target.to_string())

    # Flag anything > 0.7 as potential leakage
    suspicious = corr_with_target[corr_with_target > 0.7]
    if len(suspicious) > 0:
        print(f'\n⚠️  HIGH CORRELATION (> 0.7) — investigate for leakage:')
        print(suspicious.to_string())
    else:
        print(f'\n✅  No features with |corr| > 0.7 — leakage check passed')

**Finding 9 — Leakage Check:** Write your finding after running. Are any features suspiciously correlated (>0.7) with a target? If yes, investigate before proceeding.

---
## 10. Key Findings Summary

> **This section is mandatory.** If you cannot write 6+ findings that directly change how you build the model, you have not done EDA — you have run code.

### Findings That Drive Modeling Decisions

1. **Grain:** [FILL IN — e.g., 'Each row = one member × one service call. Must aggregate costs before dedup.']
2. **Population:** [FILL IN — e.g., 'X% of members are CANCELLED — excluded. Active population = Y households.']
3. **Modelable products:** [FILL IN — e.g., 'INS Client (17.6%), TRV Globalware (9.2%), FSV Credit Card (6.0%) exceed the 5% threshold.']
4. **ERS Signal:** [FILL IN — e.g., 'Members who used ERS show 2.3× higher INS Client adoption.']
5. **Income Signal:** [FILL IN — e.g., 'FSV Credit Card buyers have median income $X higher than non-buyers.']
6. **Cost Distribution:** [FILL IN — e.g., 'X% of members have zero ERS cost. Log transform improves normality.']
7. **Leakage:** [FILL IN — e.g., 'No features with |corr| > 0.7 to any target. Excluded columns are correct.']
8. **Multi-product opportunity:** [FILL IN — e.g., 'X% of households hold zero financial products — primary cross-sell target.']